In [3]:
import os
import cassio
from langchain_astradb import AstraDBVectorStore
from langchain_ollama import OllamaEmbeddings


os.environ['ASTRA_DB_ENDPOINT'] = os.getenv('ASTRA_DB_ENDPOINT')
os.environ['ASTRA_DB_API_TOKEN'] = os.getenv('ASTRA_DB_API_TOKEN')
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

embeddings = OllamaEmbeddings(model='nomic-embed-text')

vstore=AstraDBVectorStore(
    collection_name = "test",
    embedding=embeddings,
    token=os.environ['ASTRA_DB_API_TOKEN'],
    api_endpoint=os.environ['ASTRA_DB_ENDPOINT']

)
print('AstraDB vector store configured!')



/home/ojass-bhardwaj/Desktop/langchain/denv/lib/python3.12/site-packages/astrapy/admin/admin.py:67: UserWarning: SSL connection reuse disabled due to a Python 3.12.[0-11] bug. This may reduce performance under certain workloads. Please upgrade to Python 3.12.12 or newer if possible.
  from astrapy.utils.api_commander import APICommander


AstraDB vector store configured!


In [4]:
from datasets import load_dataset

philo_dataset = load_dataset("datastax/philosopher-quotes")["train"]
print('An example story')
print(philo_dataset[20])

An example story
{'author': 'aristotle', 'quote': 'A promise made must be a promise kept.', 'tags': 'ethics'}


In [5]:
from langchain_classic.schema import Document

docs = []
for entry in philo_dataset:
    metadata = {"author": entry["author"]}
    if entry["tags"]:
        for tag in entry["tags"].split(";"):
            metadata[tag] = "y"

    doc = Document(page_content=entry["quote"], metadata=metadata)
    docs.append(doc)
docs

[Document(metadata={'author': 'aristotle', 'knowledge': 'y'}, page_content="True happiness comes from gaining insight and growing into your best possible self. Otherwise all you're having is immediate gratification pleasure, which is fleeting and doesn't grow you as a person."),
 Document(metadata={'author': 'aristotle', 'education': 'y', 'knowledge': 'y'}, page_content='The roots of education are bitter, but the fruit is sweet.'),
 Document(metadata={'author': 'aristotle', 'ethics': 'y'}, page_content='Before you heal the body you must first heal the mind'),
 Document(metadata={'author': 'aristotle', 'education': 'y', 'knowledge': 'y'}, page_content='The proof that you know something is that you are able to teach it'),
 Document(metadata={'author': 'aristotle'}, page_content='Those who are not angry at the things they should be angry at are thought to be fools, and so are those who are not angry in the right way, at the right time, or with the right persons.'),
 Document(metadata={'au

In [6]:
inserted_ids = vstore.add_documents(docs)
print(f"\nInserted {len(inserted_ids)} documents.")


Inserted 450 documents.


In [7]:
print(list(vstore.astra_env.collection.find()))

[{'_id': '940f6cd8b20d4ee586f3322de6fb17ec', 'content': 'Just as one spoils the stomach by overfeeding and thereby impairs the whole body, so can one overload and choke the mind by giving it too much nourishment. For the more one reads the fewer are the traces left of what one has read; the mind is like a tablet that has been written over and over. Hence it is impossible to reflect; and it is only by reflection that one can assimilate what one has read. If one reads straight ahead without pondering over it later, what has been read does not take root, but is for the most part lost.', 'metadata': {'author': 'schopenhauer', 'knowledge': 'y', 'ethics': 'y'}}, {'_id': 'ac66af01e7534058849d341532538166', 'content': 'Necessity is the constant scourge of the lower classes, ennui of the higher ones.', 'metadata': {'author': 'schopenhauer'}}, {'_id': 'f5698b08977a434a84dae4366d309be8', 'content': 'Every human endeavor, however singular it seems, involves the whole human race.', 'metadata': {'au

In [ ]:
import os
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()


model = ChatGroq(
    model="llama-3.1-8b-instant",
    groq_api_key=os.getenv("GROQ_API_KEY")
)

retriever = vstore.as_retriever(search_kwargs={"k": 3})

prompt_template = """
Answer the question based only on the supplied context. If you don't know the answer, say you don't know the answer.

Context: {context}
Question: {question}

Your answer:
"""
prompt = ChatPromptTemplate.from_template(prompt_template)

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

response = chain.invoke("In the given context what is the most important to allow the brain and provide me the tags?")
print(response)

The most important thing to allow the brain is the full measure of sleep required to restore it. 

Tags: sleep, brain, restoration, winding up, clock.
